# Como uso regressão logística em séries temporais?

Para usar Regressão Logística em séries temporais, não mudamos o modelo.

Mudamos o formato dos dados.

A ideia é:

“ensinar o modelo a enxergar o passado dentro das features”

Para usar Regressão Logística em séries temporais, não mudamos o modelo.

Mudamos o formato dos dados.

A ideia é:

“ensinar o modelo a enxergar o passado dentro das features”

Isso é feito através de três estratégias principais:

1. Variáveis de Defasagem (Lags)

Ideia:

Incluir valores passados das variáveis mais importantes.

Exemplo:
| close_mid | close_mid_lag1 |
| --------: | -------------- |
|      0.45 | 0.44           |

Agora o modelo aprende:

“o preço estava subindo ou caindo recentemente?”

📌 Por que isso funciona?

Porque o mercado financeiro possui dependência de curto prazo.

Movimentos recentes influenciam fortemente o próximo movimento.

2. Variáveis de microestrutura com lag

Aplicamos a mesma ideia para variáveis importantes:

depth_imbalance
mean_spread
bar_volatility
ofi_corrected

Isso permite que o modelo capture:

pressão compradora recente
liquidez recente
volatilidade recente

# Objetivo:
O objetivo é enriquecer o dataset com informações que representem melhor a dinâmica temporal dos mercados da Polymarket, permitindo que modelos tradicionais de classificação, como Regressão Logística, Árvore de Decisão e Random Forest, utilizem informações do histórico recente durante o treinamento.

Além disso, será corrigida a variável order_flow_imbalance, cuja implementação original apresentou inconsistências identificadas durante a Análise Exploratória dos Dados.

As novas variáveis serão adicionadas em uma nova versão do dataset, preservando o arquivo original.

In [1]:
import polars as pl

# Carrega o dataset original
df = pl.read_parquet("features/ml_features_1m_v2.parquet")

In [2]:
df.columns

['market_id',
 'minute_bar',
 'close_mid',
 'mean_spread',
 'close_spread',
 'bar_volatility',
 'total_volume',
 'buy_volume',
 'sell_volume',
 'trade_count',
 'order_flow_imbalance',
 'target',
 'return_1m',
 'bid_depth',
 'ask_depth',
 'depth_imbalance']

# Ordenação dos dados:
Antes de criar qualquer variável temporal precisamos garantir que os registros estejam na ordem correta.

Por quê?
Cada linha representa um minuto de um determinado mercado.

Imagine este exemplo:
| market_id | minute_bar | close_mid |
| --------- | ---------- | --------: |
| A         | 10:00      |      0.42 |
| A         | 10:01      |      0.43 |
| A         | 10:02      |      0.44 |

Agora imagine outro mercado:
| market_id | minute_bar | close_mid |
| --------- | ---------- | --------: |
| B         | 09:00      |      0.71 |
| B         | 09:01      |      0.70 |

Se os dados não estiverem ordenados por market_id e minute_bar, existe o risco de associar informações de mercados diferentes, produzindo features incorretas.

In [3]:
df = df.sort(
    ["market_id", "minute_bar"]
)

# Correção do order flow imbalance

In [6]:
df = df.with_columns(
    pl.when(pl.col("total_volume") > 0)
    .then(
        (
            pl.col("buy_volume") -
            pl.col("sell_volume")
        ) / pl.col("total_volume")
    )
    .otherwise(0.0)
    .alias("ofi_corrected")
)

df.select(
    pl.min("ofi_corrected").alias("min"),
    pl.max("ofi_corrected").alias("max"),
    pl.mean("ofi_corrected").alias("mean")
)

min,max,mean
f32,f32,f32
-1.0,1.0,0.026979


# Criação de Variáveis de Defasagem (Lag Features)

Adicionar ao dataset informações sobre o comportamento recente de cada mercado.

Modelos tradicionais de aprendizado de máquina, como Regressão Logística, Árvore de Decisão e Random Forest, tratam cada linha do dataset como uma observação independente. Entretanto, em problemas de séries temporais, as observações possuem uma relação cronológica, onde eventos passados podem influenciar eventos futuros.

Para incorporar essa dependência temporal ao modelo, serão criadas variáveis de defasagem (lag features), que armazenam o valor da variável observado em instantes anteriores.

Considere um mesmo mercado ao longo do tempo:
| Horário | close_mid |
| ------- | --------: |
| 10:00   |      0.45 |
| 10:01   |      0.47 |
| 10:02   |      0.49 |
| 10:03   |      0.50 |

Sem utilizar lags o modelo observa apenas o preço atual (0,50)

Com os lags de 1 minuto, o modelo passa a observar:
| close_mid | close_mid_lag1 |
| --------- | -------------- |
| 0.50      | 0.49           |

Agora o modelo sabe: "há um minuto o preço era 0.49 e agora é 0.50."

Essa informação representa a evolução recente do mercado e pode auxiliar na previsão do comportamento futuro.

Por que utilizaremos apenas Lag 1?

Existem diversas possibilidades, como:

Lag 1;
Lag 2;
Lag 5;
Lag 10.

Neste projeto será utilizado apenas Lag 1, pelos seguintes motivos:

representa o estado imediatamente anterior do mercado;
mantém o modelo simples e interpretável;
reduz a quantidade de atributos criados;
evita adicionar variáveis altamente correlacionadas entre si.

Caso seja necessário aumentar a complexidade do modelo futuramente, outros lags poderão ser incorporados.

### Quais variáveis receberão Lag?

Essa decisão foi baseada na Análise Exploratória dos Dados.

Serão utilizadas apenas variáveis que demonstraram maior relação com o target.

| Variável          | Justificativa                                 |
| ----------------- | --------------------------------------------- |
| `close_mid`       | Principal indicador de preço do contrato.     |
| `depth_imbalance` | Forte relação com a direção futura do preço.  |
| `mean_spread`     | Representa a liquidez média do mercado.       |
| `close_spread`    | Indica a liquidez no encerramento do minuto.  |
| `bar_volatility`  | Mede a intensidade das oscilações de preço.   |
| `ofi_corrected`   | Representa a pressão compradora ou vendedora. |


In [8]:
lag_features = [
    "close_mid",
    "depth_imbalance",
    "mean_spread",
    "close_spread",
    "bar_volatility",
    "ofi_corrected"
]

df = df.with_columns(
    [
        pl.col(col)
        .shift(1)
        .over("market_id")
        .alias(f"{col}_lag1")
        for col in lag_features
    ]
)

In [9]:
df.select(
    [
        "market_id",
        "minute_bar",
        "close_mid",
        "close_mid_lag1",
        "ofi_corrected",
        "ofi_corrected_lag1"
    ]
).head(10)

market_id,minute_bar,close_mid,close_mid_lag1,ofi_corrected,ofi_corrected_lag1
str,"datetime[μs, UTC]",f32,f32,f32,f32
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:00:00 UTC,0.285,null,0.0,null
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:01:00 UTC,0.285,0.285,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:02:00 UTC,0.29,0.285,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:03:00 UTC,0.29,0.29,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:04:00 UTC,0.29,0.29,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:05:00 UTC,0.27,0.29,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:06:00 UTC,0.275,0.27,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:07:00 UTC,0.275,0.275,0.0,0.0
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:08:00 UTC,0.28,0.275,0.0,0.0


In [10]:
df.columns

['market_id',
 'minute_bar',
 'close_mid',
 'mean_spread',
 'close_spread',
 'bar_volatility',
 'total_volume',
 'buy_volume',
 'sell_volume',
 'trade_count',
 'order_flow_imbalance',
 'target',
 'return_1m',
 'bid_depth',
 'ask_depth',
 'depth_imbalance',
 'ofi_corrected',
 'close_mid_lag1',
 'depth_imbalance_lag1',
 'mean_spread_lag1',
 'close_spread_lag1',
 'bar_volatility_lag1',
 'ofi_corrected_lag1']

# Validação das features

Validação dos lags. Vou verificar se:

feature_lag1 = valor da linha imediatamente anterior do mesmo mercado

In [13]:
df.filter(
    pl.col("market_id") == df["market_id"][0]
).select([
    "market_id",
    "minute_bar",
    "close_mid",
    "close_mid_lag1",
    "depth_imbalance",
    "depth_imbalance_lag1"
]).head(15)

market_id,minute_bar,close_mid,close_mid_lag1,depth_imbalance,depth_imbalance_lag1
str,"datetime[μs, UTC]",f32,f32,f64,f64
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:00:00 UTC,0.285,null,-0.860731,null
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:01:00 UTC,0.285,0.285,-0.860731,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:02:00 UTC,0.29,0.285,-0.860731,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:03:00 UTC,0.29,0.29,-0.860731,-0.860731
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:04:00 UTC,0.29,0.29,-0.92031,-0.860731
…,…,…,…,…,…
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:10:00 UTC,0.28,0.28,-0.92031,-0.92031
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:11:00 UTC,0.265,0.28,-0.92031,-0.92031
"""0x0007deb167d0bb816e2e847a1543…",2026-03-06 00:12:00 UTC,0.265,0.265,-0.92031,-0.92031


Verificação dos nulls.

É esperado que a primeira linha de cada feature seja null, pois não havia nenhum dado antes dela.
Então, como temos 4710 mercados, devemos ter 4710 linhas com null (que é a primeira de cada um deles)

Essas linhas podem ser:

removidas antes do treino
ou mantidas dependendo do modelo

para regressão logística: geralmente removemos


In [15]:
df.select(
    [
        pl.col("close_mid_lag1").null_count().alias("null_close_mid_lag1"),
        pl.col("ofi_corrected_lag1").null_count().alias("null_ofi_lag1"),
        pl.col("depth_imbalance_lag1").null_count().alias("null_depth_imbalance_lag1")
    ]
)

null_close_mid_lag1,null_ofi_lag1,null_depth_imbalance_lag1
u32,u32,u32
4710,4710,4710


Validação estatística:

Tem que fazer sentido com o que deu com as features normais. No resultado deu tudo certo

In [16]:
df.select([
    pl.mean("close_mid_lag1"),
    pl.mean("depth_imbalance_lag1"),
    pl.mean("mean_spread_lag1"),
    pl.mean("ofi_corrected_lag1")
])

close_mid_lag1,depth_imbalance_lag1,mean_spread_lag1,ofi_corrected_lag1
f32,f64,f32,f32
0.264241,-0.337331,0.106515,0.026992


Validação da consistência por mercado:  

In [17]:
df.group_by("market_id").len().describe()

statistic,market_id,len
str,str,f64
"""count""","""4710""",4710.0
"""null_count""","""0""",0.0
"""mean""",null,1186.315711
"""std""",null,1314.819778
"""min""","""0x0007deb167d0bb816e2e847a1543…",1.0
"""25%""",null,296.0
"""50%""",null,662.0
"""75%""",null,1588.0
"""max""","""0xfff145c6201796960cac69aea3f6…",8221.0


In [26]:
print(df.shape)

(5587547, 23)


In [ ]:
df.write_parquet(
    "features/ml_features_v3.parquet"
)